# Chatbot RAG tư vấn bất động sản (bản học tập)

Chạy lần lượt từng ô từ trên xuống bằng **Shift + Enter**.

Đặt notebook này cùng thư mục với 3 file: `tin_dang.csv`, `du_an.md`, `hoi_dap_mua_nha.md`.

## Bước 1: Cài thư viện

Chỉ cần chạy một lần. Sau khi cài xong, có thể phải khởi động lại kernel (Kernel > Restart).

In [1]:
%pip install -q anthropic chromadb sentence-transformers


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import csv
import os
import re
from getpass import getpass

import anthropic
import chromadb
from chromadb.utils import embedding_functions

MODEL = "claude-sonnet-5-5"  # đổi sang model khác nếu muốn rẻ hơn
SO_DOAN_LAY_RA = 3           # số đoạn tài liệu đưa cho Claude mỗi câu hỏi

## Bước 2: Cắt tài liệu thành đoạn

- File `.md`: mỗi mục bắt đầu bằng `##` là một đoạn.
- File `.csv`: mỗi dòng tin đăng là một đoạn, viết lại thành câu văn.

In [3]:
def doc_markdown(duong_dan):
    """Mỗi mục bắt đầu bằng '## ' trong file .md thành một đoạn."""
    with open(duong_dan, encoding="utf-8") as f:
        noi_dung = f.read()

    cac_doan = []
    # Tách theo dòng bắt đầu bằng "## ", bỏ phần trước mục đầu tiên
    for phan in re.split(r"^## ", noi_dung, flags=re.MULTILINE)[1:]:
        tieu_de, _, than_bai = phan.partition("\n")
        cac_doan.append({
            "text": f"{tieu_de.strip()}\n{than_bai.strip()}",
            "nguon": f"{duong_dan} > {tieu_de.strip()}",
        })
    return cac_doan


def doc_csv(duong_dan):
    """Mỗi dòng tin đăng thành một đoạn, viết lại thành câu văn."""
    ten_loai = {"can_ho": "Căn hộ", "nha_pho": "Nhà phố", "dat_nen": "Đất nền"}

    cac_doan = []
    with open(duong_dan, encoding="utf-8") as f:
        for dong in csv.DictReader(f):
            du_an = f", thuộc dự án {dong['du_an']}" if dong["du_an"] else ""
            text = (
                f"Tin đăng {dong['id']}: {dong['tieu_de']}. "
                f"Loại: {ten_loai.get(dong['loai'], dong['loai'])}{du_an}. "
                f"Vị trí: {dong['quan']}, {dong['thanh_pho']}. "
                f"Giá: {dong['gia_ty']} tỷ đồng. "
                f"Diện tích: {dong['dien_tich_m2']} m2. "
                f"{dong['phong_ngu']} phòng ngủ, {dong['phong_tam']} phòng tắm. "
                f"Pháp lý: {dong['phap_ly']}. "
                f"Ngày đăng: {dong['ngay_dang']}. "
                f"Mô tả: {dong['mo_ta']}"
            )
            cac_doan.append({"text": text, "nguon": f"{duong_dan} > {dong['id']}"})
    return cac_doan


cac_doan = (
    doc_markdown("du_an.md")
    + doc_markdown("hoi_dap_mua_nha.md")
    + doc_csv("tin_dang.csv")
)
print(f"Tổng cộng {len(cac_doan)} đoạn")

Tổng cộng 22 đoạn


Xem danh sách các đoạn vừa cắt:

In [4]:
for i, d in enumerate(cac_doan):
    print(f"{i:>2}  {len(d['text']):>4} ký tự  {d['nguon']}")

 0   740 ký tự  du_an.md > Sông Xanh Residence
 1   844 ký tự  du_an.md > An Lạc Garden
 2   569 ký tự  du_an.md > Thiên Phú Tower
 3   457 ký tự  hoi_dap_mua_nha.md > Quy trình mua một căn nhà đã có sổ gồm những bước nào?
 4   384 ký tự  hoi_dap_mua_nha.md > Đặt cọc bao nhiêu và nếu đổi ý thì sao?
 5   324 ký tự  hoi_dap_mua_nha.md > Khi mua nhà phải trả những khoản thuế phí nào?
 6   411 ký tự  hoi_dap_mua_nha.md > Sổ hồng và hợp đồng mua bán khác nhau thế nào?
 7   373 ký tự  hoi_dap_mua_nha.md > Vay ngân hàng mua nhà được tối đa bao nhiêu?
 8   356 ký tự  hoi_dap_mua_nha.md > Nên kiểm tra gì trước khi mua căn hộ dự án chưa bàn giao?
 9   338 ký tự  hoi_dap_mua_nha.md > Mua nhà để cho thuê thì tính lợi nhuận thế nào?
10   323 ký tự  tin_dang.csv > T001
11   318 ký tự  tin_dang.csv > T002
12   309 ký tự  tin_dang.csv > T003
13   316 ký tự  tin_dang.csv > T004
14   307 ký tự  tin_dang.csv > T005
15   317 ký tự  tin_dang.csv > T006
16   279 ký tự  tin_dang.csv > T007
17   264 ký tự  ti

Xem nội dung đầy đủ của một đoạn bất kỳ (đổi số trong ngoặc vuông để xem đoạn khác):

In [5]:
print(cac_doan[0]["text"])

Sông Xanh Residence
Sông Xanh Residence là khu căn hộ tại Quận 7, TP.HCM, do Công ty Cổ phần Phát triển Nhà Sông Xanh làm chủ đầu tư. Dự án gồm 3 tòa tháp cao 25 tầng với tổng cộng 780 căn hộ, đã bàn giao từ tháng 6/2023 và đã cấp sổ hồng cho cư dân.

Tiện ích nội khu gồm hồ bơi 50m, phòng gym, khu vui chơi trẻ em, công viên ven sông dài 300m và siêu thị mini ở tầng trệt. Dự án cách trường quốc tế 500m, cách trung tâm thương mại 1,5km và cách trung tâm Quận 1 khoảng 20 phút đi ô tô.

Phí quản lý là 16.000 đồng/m2/tháng. Phí gửi xe máy 150.000 đồng/tháng, ô tô 1.500.000 đồng/tháng. Mỗi căn 3 phòng ngủ được ưu tiên đăng ký một chỗ đậu ô tô.

Điểm cần lưu ý: khu vực đường vào dự án thường kẹt xe giờ cao điểm buổi sáng từ 7h đến 8h30.


## Bước 3: Tạo embedding và lưu vào Chroma

Lần chạy đầu sẽ tải embedding model về (khoảng 500MB) nên hơi lâu.

In [6]:
# Embedding model đa ngôn ngữ, chạy ngay trên máy, miễn phí
embedding = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-MiniLM-L12-v2"
)

client = chromadb.Client()  # lưu trong bộ nhớ, tắt kernel là mất

# Xoá kho cũ nếu có, để chạy lại ô này nhiều lần không bị lỗi trùng tên
try:
    client.delete_collection("bat_dong_san")
except Exception:
    pass

kho = client.create_collection("bat_dong_san", embedding_function=embedding)

# Chroma tự gọi embedding model cho từng đoạn khi mình add vào
kho.add(
    ids=[str(i) for i in range(len(cac_doan))],
    documents=[d["text"] for d in cac_doan],
    metadatas=[{"nguon": d["nguon"]} for d in cac_doan],
)
print(f"Kho có {kho.count()} đoạn")

/Users/nguyenthanhngochan14122589gmail.com/Desktop/chat_box_property/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 16040.48it/s]


Kho có 22 đoạn


Xem thử một vector trông như thế nào:

In [7]:
vector = embedding(["thủ tục sang tên sổ đỏ"])[0]
print(f"Vector có {len(vector)} con số. 8 số đầu tiên:")
print([round(float(x), 3) for x in vector[:8]])

Vector có 384 con số. 8 số đầu tiên:
[-0.137, 0.123, -0.179, -0.072, -0.155, 0.038, 0.247, 0.065]


In [ ]:
print(cac_doan[0]["text"])

Sông Xanh Residence
Sông Xanh Residence là khu căn hộ tại Quận 7, TP.HCM, do Công ty Cổ phần Phát triển Nhà Sông Xanh làm chủ đầu tư. Dự án gồm 3 tòa tháp cao 25 tầng với tổng cộng 780 căn hộ, đã bàn giao từ tháng 6/2023 và đã cấp sổ hồng cho cư dân.

Tiện ích nội khu gồm hồ bơi 50m, phòng gym, khu vui chơi trẻ em, công viên ven sông dài 300m và siêu thị mini ở tầng trệt. Dự án cách trường quốc tế 500m, cách trung tâm thương mại 1,5km và cách trung tâm Quận 1 khoảng 20 phút đi ô tô.

Phí quản lý là 16.000 đồng/m2/tháng. Phí gửi xe máy 150.000 đồng/tháng, ô tô 1.500.000 đồng/tháng. Mỗi căn 3 phòng ngủ được ưu tiên đăng ký một chỗ đậu ô tô.

Điểm cần lưu ý: khu vực đường vào dự án thường kẹt xe giờ cao điểm buổi sáng từ 7h đến 8h30.


## Bước 4: Tìm kiếm

Ô này **chưa dùng Claude**. Kết quả in kèm *khoảng cách*: số càng nhỏ thì đoạn càng gần nghĩa với câu hỏi.

Nên thử kỹ ở đây trước: nếu đoạn đúng không xuất hiện thì Claude cũng không trả lời đúng được.

In [8]:
def tim_kiem(cau_hoi, k=SO_DOAN_LAY_RA):
    """Trả về k đoạn có nghĩa gần câu hỏi nhất."""
    ket_qua = kho.query(query_texts=[cau_hoi], n_results=k)
    return [
        {"text": text, "nguon": meta["nguon"], "khoang_cach": kc}
        for text, meta, kc in zip(
            ket_qua["documents"][0], ket_qua["metadatas"][0], ket_qua["distances"][0]
        )
    ]


def xem_tim_kiem(cau_hoi, k=SO_DOAN_LAY_RA):
    print(f"Câu hỏi: {cau_hoi}")
    for d in tim_kiem(cau_hoi, k):
        print(f"  {d['khoang_cach']:.3f}  {d['nguon']}")
    print()


xem_tim_kiem("Mua nhà phải đóng những loại thuế phí gì?")
xem_tim_kiem("Dự án nào gần metro?")
xem_tim_kiem("An Lạc Garden thanh toán thế nào?")

Câu hỏi: Mua nhà phải đóng những loại thuế phí gì?
  0.152  hoi_dap_mua_nha.md > Khi mua nhà phải trả những khoản thuế phí nào?
  0.364  hoi_dap_mua_nha.md > Đặt cọc bao nhiêu và nếu đổi ý thì sao?
  0.432  hoi_dap_mua_nha.md > Mua nhà để cho thuê thì tính lợi nhuận thế nào?

Câu hỏi: Dự án nào gần metro?
  0.619  tin_dang.csv > T007
  0.619  tin_dang.csv > T005
  0.621  tin_dang.csv > T008

Câu hỏi: An Lạc Garden thanh toán thế nào?
  0.414  du_an.md > An Lạc Garden
  0.570  tin_dang.csv > T006
  0.646  hoi_dap_mua_nha.md > Khi mua nhà phải trả những khoản thuế phí nào?



## Bước 5: Nối Claude

Ô dưới sẽ hỏi API key nếu máy chưa có biến môi trường `ANTHROPIC_API_KEY`. Key được gõ ẩn và không lưu vào notebook.

In [ ]:
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Dán API key của Claude: ")

claude = anthropic.Anthropic()

In [ ]:
CHI_DAN = """Bạn là trợ lý tư vấn bất động sản.
Chỉ trả lời dựa trên các đoạn tài liệu được cung cấp trong thẻ <tai_lieu>.
Nếu tài liệu không có thông tin để trả lời, hãy nói rõ là bạn không có thông tin đó, không được tự suy đoán.
Cuối câu trả lời, ghi nguồn đã dùng theo dạng [Nguồn: ...].
Trả lời ngắn gọn bằng tiếng Việt."""


def hoi(cau_hoi, k=SO_DOAN_LAY_RA):
    """Tìm các đoạn liên quan, gửi cho Claude, in ra cả hai."""
    tim_duoc = tim_kiem(cau_hoi, k)

    print(f"Câu hỏi: {cau_hoi}\n")
    print("Đoạn tìm được:")
    for d in tim_duoc:
        print(f"  {d['khoang_cach']:.3f}  {d['nguon']}")

    tai_lieu = "\n\n".join(
        f"<tai_lieu nguon=\"{d['nguon']}\">\n{d['text']}\n</tai_lieu>" for d in tim_duoc
    )
    phan_hoi = claude.messages.create(
        model=MODEL,
        max_tokens=1024,
        system=CHI_DAN,
        messages=[{"role": "user", "content": f"{tai_lieu}\n\nCâu hỏi: {cau_hoi}"}],
    )
    print(f"\nChatbot: {phan_hoi.content[0].text}")

## Bước 6: Thử 5 câu hỏi kiểm tra

Mỗi câu một ô để dễ so sánh kết quả.

In [ ]:
hoi("Mua nhà phải đóng những loại thuế phí gì?")

In [ ]:
hoi("Dự án nào gần metro?")

In [ ]:
hoi("An Lạc Garden thanh toán thế nào?")

In [ ]:
hoi("Căn 2 phòng ngủ dưới 3 tỷ ở Quận 7?")

In [ ]:
hoi("Giá nhà ở Hà Nội thế nào?")

Tự hỏi câu của mình:

In [ ]:
hoi("Viết câu hỏi của chị vào đây")